# Neural Network Model

In [1]:
# import relevant libraries
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    precision_recall_fscore_support,
    accuracy_score,
    log_loss,
 )
from sklearn.neural_network import MLPClassifier
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

from tqdm.auto import tqdm

In training the neural network, TF-IDF of the word and character n-grams will be used as the feature to be fed throught the model. As mentioned before in the corpus analysis notebook, TF-IDF can be used as a metric to be used for classifying text authorship. For the model architecture, both shallow and deep neural network architecture will be explored. We are curious to see if a simpler architecture is more fit considering the small dataset, and the limited authors we have. Prior works have tried to use both architectures for the models [1,2,3].

The model will utilize ReLU activation functions to improve training efficiency and mitigate vanishing gradient issues [4]. The Adam optimizer will be used due to its adaptive learning rate and strong empirical performance across a wide range of machine learning tasks [5] Additionally, early stopping is applied to prevent overfitting by monitoring validation performance during training [6].

## Data prep and TF-IDF features

In [2]:
DATA_PATH = Path("llm_data/llm-dataset/eli5_all_llm_answers_cleaned.csv")
df = pd.read_csv(DATA_PATH)

long_df = df.melt(
    id_vars = ["q_id", "question"],
    value_vars = ["chatgpt", "deepseek", "gemini"],
    var_name = "author",
    value_name = "text",
).dropna(subset = ["text"])
label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(long_df["author"])
texts = long_df["text"].astype(str).tolist()


### Data Split

In [3]:
# 70-15-15 split for train, validation, and test sets
X_train_texts, X_temp_texts, y_train, y_temp = train_test_split(
    texts, 
    labels, 
    test_size = 0.3, 
    random_state = 67, 
    stratify = labels
)
X_val_texts, X_test_texts, y_val, y_test = train_test_split(
    X_temp_texts, 
    y_temp, 
    test_size = 0.5, 
    random_state = 67, 
    stratify = y_temp
)

In [4]:
# Inspect training data and labels
train_df = pd.DataFrame({"text": X_train_texts, "label_id": y_train})
train_df["label_name"] = label_encoder.inverse_transform(train_df["label_id"])

print("Training label distribution:")
print(train_df["label_name"].value_counts())

print("\nSample training rows:")
display(train_df.head(20))

Training label distribution:
label_name
deepseek    9288
chatgpt     9288
gemini      9288
Name: count, dtype: int64

Sample training rows:


,text,label_id,label_name
0,That note is there because coupons are like li...,1,deepseek
1,Pi-hole is like a superhero for your internet!...,0,chatgpt
2,"Sometimes when people are really excited, nerv...",1,deepseek
3,"No, you can't make a bigger purchase just beca...",0,chatgpt
4,Imagine your brain is like a superhero that so...,2,gemini
5,"No, you cannot keep an electron completely sep...",1,deepseek
6,"A single payer plan is like if one big, friend...",0,chatgpt
7,"Okay, imagine your body is like a big LEGO cas...",0,chatgpt
8,"That weird smell happens because tiny, invisib...",1,deepseek
9,"Before people could buy plane tickets online, ...",0,chatgpt


In [5]:
# Inspect validation data and labels
val_df = pd.DataFrame({"text": X_val_texts, "label_id": y_val})
val_df["label_name"] = label_encoder.inverse_transform(val_df["label_id"])

print("Validation label distribution:")
print(val_df["label_name"].value_counts())

print("\nSample validation rows:")
display(val_df.sample(min(5, len(val_df)), random_state=67))



Validation label distribution:
label_name
gemini      1991
deepseek    1990
chatgpt     1990
Name: count, dtype: int64

Sample validation rows:


,text,label_id,label_name
4278,A shirt dries when water evaporates because wa...,1,deepseek
2287,"Okay, imagine the air all around us, even at n...",2,gemini
4593,"Think of the Middle East like a very old, busy...",1,deepseek
1176,"Okay, imagine you have a magic box of LEGO bri...",0,chatgpt
1961,Think of a mirror like a window into a copy of...,1,deepseek


### Getting the n-grams

In [ ]:
word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range = (2, 3),  # word bigram and trigram 
    min_df = 10, # increase min_df to reduce noise and dimensionality
    max_features = 3072, # limit to avoid overfitting and for computational efficiency
    dtype = np.float32,
    )

char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range = (3, 4), # char 3-grams to 4-grams
    min_df = 10, # increase min_df to reduce noise and dimensionality
    max_features = 3072,  # limit to avoid overfitting and for computational efficiency
    dtype = np.float32,
    )

X_train_word = word_vectorizer.fit_transform(X_train_texts)
X_val_word = word_vectorizer.transform(X_val_texts)
X_test_word = word_vectorizer.transform(X_test_texts)

X_train_char = char_vectorizer.fit_transform(X_train_texts)
X_val_char = char_vectorizer.transform(X_val_texts)
X_test_char = char_vectorizer.transform(X_test_texts)

num_features_word = X_train_word.shape[1]
num_features_char = X_train_char.shape[1]
num_classes = len(label_encoder.classes_)

For this, we do not need feature scaling anymore since we are using TF-IDF. The max_features value is set based on previous runs of the training. We started it at 16384, and lowered it to lessen overfitting until we found a good value.

## Grid search for hidden layer sizes

### Globals


We will be using standard learning rates used in neural network training. We will also incorporate L2 regularization to lessen the overfitting of the data.

In [7]:
lr_search_space = [1e-4, 2e-4]
l2_search_space = [1e-3, 5e-4]

best_config_by_feature = {}
best_val_acc_by_feature = {}
results_by_feature = {}
histories_by_feature = {}


### Helper Functions

For this project, we will be using scikit's multi-layer perceptron (MLP) classifier. This classifier can be used for developing image recognition, and text classification [7].

In [8]:
# Training function for config
def fit_mlp_classifier(X_train, y_train, X_val, y_val, hidden_neurons, lr, max_iter, patience, l2_reg):
    model = MLPClassifier(
        hidden_layer_sizes = hidden_neurons,
        activation = "relu",
        solver = "adam",
        batch_size = 32,
        learning_rate_init = lr,
        max_iter = 1,
        warm_start = True,
        shuffle = True,
        early_stopping = False,
        random_state = 67,
        alpha = l2_reg
    )

    history = []
    best_val_loss = float("inf")
    epochs_no_improve = 0
    labels = np.unique(y_train)

    for epoch in range(1, max_iter + 1):
        model.fit(X_train, y_train)
        train_loss = model.loss_
        train_preds = model.predict(X_train)
        train_acc = accuracy_score(y_train, train_preds)
        val_probs = model.predict_proba(X_val)
        val_loss = log_loss(y_val, val_probs, labels=labels)
        val_preds = model.predict(X_val)
        val_acc = accuracy_score(y_val, val_preds)

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
        })

        print(
            f"Epoch {epoch:03d} | "
            f"train_loss={train_loss:.4f} "
            f"train_acc={train_acc:.4f} "
            f"val_loss={val_loss:.4f} "
            f"val_acc={val_acc:.4f}"
        )
        # subtract patience if val_loss does not improve by at least 0.001 
        if val_loss < best_val_loss - 1e-3: 
            best_val_loss = val_loss
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break
    return model, val_acc, history

# Trying all configs and find which one is the best via the validation accuracy
def run_grid_search(feature_name, cfg):
    best_config = None
    best_val_acc = 0.0
    results = []
    histories = []

    for params in ParameterGrid({
        "hidden_neurons": cfg["hidden_neurons"],
        "lr": lr_search_space,
        "l2_reg": l2_search_space,
    }):
        hidden_neurons = params["hidden_neurons"]
        lr = params["lr"]
        l2_reg = params["l2_reg"]

        print(
            f"\n[{feature_name}] Training model with "
            f"hidden_neurons={hidden_neurons}, lr={lr}, l2_reg={l2_reg}"
        )

        model, val_acc, history = fit_mlp_classifier(
            cfg["X_train"],
            cfg["y_train"],
            cfg["X_val"],
            cfg["y_val"],
            hidden_neurons = hidden_neurons,
            lr = lr,
            max_iter = 200,
            patience = 3,
            l2_reg = l2_reg,
        )

        results.append({
            "hidden_neurons": hidden_neurons,
            "lr": lr,
            "l2_reg": l2_reg,
            "val_acc": val_acc,
        })

        histories.append({
            "hidden_neurons": hidden_neurons,
            "lr": lr,
            "l2_reg": l2_reg,
            "history": history,
        })

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_config = {
                "hidden_neurons": hidden_neurons,
                "lr": lr,
                "l2_reg": l2_reg,
            }

    return best_config, best_val_acc, results, histories

# Self-explanatory
def final_evaluation(model, X_test, y_test, target_names):
    preds = model.predict(X_test)
    report = classification_report(
        y_test,
        preds,
        target_names = target_names,
        digits = 4,
        zero_division = 0,
    )
    return report

# Train with the best config found
def train_best_for_feature(feature, cfg, X_test, y_test):
    best_cfg = best_config_by_feature.get(feature)

    if best_cfg is None:
        raise RuntimeError(f"best_config not found for feature: {feature}")
    
    model, val_acc, history = fit_mlp_classifier(
        cfg["X_train"],
        cfg["y_train"],
        cfg["X_val"],
        cfg["y_val"],
        hidden_neurons = best_cfg["hidden_neurons"],
        lr = best_cfg["lr"],
        max_iter = 200,
        patience = 3,
        l2_reg = best_cfg["l2_reg"],
    )

    report = final_evaluation(model, X_test, y_test, target_names=label_encoder.classes_.tolist())
    return model, report

### Data Prep

The MLPClassifier that we are going to use works better with dense input.

In [9]:
X_train_word_dense = X_train_word.toarray().astype(np.float32)
X_val_word_dense = X_val_word.toarray().astype(np.float32)
X_test_word_dense = X_test_word.toarray().astype(np.float32)

X_train_char_dense = X_train_char.toarray().astype(np.float32)
X_val_char_dense = X_val_char.toarray().astype(np.float32)
X_test_char_dense = X_test_char.toarray().astype(np.float32)

### Variables for Grid Search

In [10]:
feature_configs = {
    "word": {
        "X_train": X_train_word_dense,
        "y_train": y_train,
        "X_val": X_val_word_dense,
        "y_val": y_val,
        "hidden_neurons": [(32,), (64,), (128,), (32, 32, 32), (64, 64, 64), (128, 128, 128)], # shallow and deep configs with 32, 64, and 128 neurons
    },
    "char": {
        "X_train": X_train_char_dense,
        "y_train": y_train,
        "X_val": X_val_char_dense,
        "y_val": y_val,
        "hidden_neurons": [(32,), (64,), (128,), (32, 32, 32), (64, 64, 64), (128, 128, 128)],
    },
}

### Grid Search Run

We do a grid search in order to find the best configuration based on the accuracy on validation set. This essentially serves as our hyperparameter tuning, as well as for determining which neural network structure works well.

In [11]:
for feature_name, cfg in feature_configs.items():
    best_cfg, best_acc, results, histories = run_grid_search(feature_name, cfg)
    best_config_by_feature[feature_name] = best_cfg
    best_val_acc_by_feature[feature_name] = best_acc
    results_by_feature[feature_name] = results
    histories_by_feature[feature_name] = histories

best_config_by_feature, best_val_acc_by_feature


[word] Training model with hidden_neurons=(32,), lr=0.0001, l2_reg=0.001
Epoch 001 | train_loss=0.9872 train_acc=0.8351 val_loss=0.8401 val_acc=0.8277
Epoch 001 | train_loss=0.9872 train_acc=0.8351 val_loss=0.8401 val_acc=0.8277
Epoch 002 | train_loss=0.7290 train_acc=0.8844 val_loss=0.6282 val_acc=0.8776
Epoch 002 | train_loss=0.7290 train_acc=0.8844 val_loss=0.6282 val_acc=0.8776
Epoch 003 | train_loss=0.5446 train_acc=0.9097 val_loss=0.4779 val_acc=0.9037
Epoch 003 | train_loss=0.5446 train_acc=0.9097 val_loss=0.4779 val_acc=0.9037
Epoch 004 | train_loss=0.4182 train_acc=0.9218 val_loss=0.3779 val_acc=0.9163
Epoch 004 | train_loss=0.4182 train_acc=0.9218 val_loss=0.3779 val_acc=0.9163
Epoch 005 | train_loss=0.3346 train_acc=0.9310 val_loss=0.3126 val_acc=0.9230
Epoch 005 | train_loss=0.3346 train_acc=0.9310 val_loss=0.3126 val_acc=0.9230
Epoch 006 | train_loss=0.2793 train_acc=0.9378 val_loss=0.2694 val_acc=0.9278
Epoch 006 | train_loss=0.2793 train_acc=0.9378 val_loss=0.2694 val_a

({'word': {'hidden_neurons': (32,), 'lr': 0.0001, 'l2_reg': 0.001},
  'char': {'hidden_neurons': (32,), 'lr': 0.0002, 'l2_reg': 0.001}},
 {'word': 0.9408809244682633, 'char': 0.975046055937029})

## Train with the best configuration, and Test with test dataset

### Best Word n-gram Model

In [12]:
feature = "word"
model, word_report = train_best_for_feature(
    feature,
    feature_configs[feature],
    X_test_word_dense,
    y_test,
 )
print(word_report)

Epoch 001 | train_loss=0.9872 train_acc=0.8351 val_loss=0.8401 val_acc=0.8277
Epoch 002 | train_loss=0.7290 train_acc=0.8844 val_loss=0.6282 val_acc=0.8776
Epoch 002 | train_loss=0.7290 train_acc=0.8844 val_loss=0.6282 val_acc=0.8776
Epoch 003 | train_loss=0.5446 train_acc=0.9097 val_loss=0.4779 val_acc=0.9037
Epoch 003 | train_loss=0.5446 train_acc=0.9097 val_loss=0.4779 val_acc=0.9037
Epoch 004 | train_loss=0.4182 train_acc=0.9218 val_loss=0.3779 val_acc=0.9163
Epoch 004 | train_loss=0.4182 train_acc=0.9218 val_loss=0.3779 val_acc=0.9163
Epoch 005 | train_loss=0.3346 train_acc=0.9310 val_loss=0.3126 val_acc=0.9230
Epoch 005 | train_loss=0.3346 train_acc=0.9310 val_loss=0.3126 val_acc=0.9230
Epoch 006 | train_loss=0.2793 train_acc=0.9378 val_loss=0.2694 val_acc=0.9278
Epoch 006 | train_loss=0.2793 train_acc=0.9378 val_loss=0.2694 val_acc=0.9278
Epoch 007 | train_loss=0.2416 train_acc=0.9436 val_loss=0.2399 val_acc=0.9312
Epoch 007 | train_loss=0.2416 train_acc=0.9436 val_loss=0.2399 v

### Best Char n-gram Model

In [13]:
feature = "char"
model, char_report = train_best_for_feature(
    feature,
    feature_configs[feature],
    X_test_char_dense,
    y_test,
 )
print(char_report)

Epoch 001 | train_loss=0.6942 train_acc=0.9334 val_loss=0.3942 val_acc=0.9338
Epoch 002 | train_loss=0.2866 train_acc=0.9526 val_loss=0.2090 val_acc=0.9538
Epoch 002 | train_loss=0.2866 train_acc=0.9526 val_loss=0.2090 val_acc=0.9538
Epoch 003 | train_loss=0.1795 train_acc=0.9626 val_loss=0.1485 val_acc=0.9608
Epoch 003 | train_loss=0.1795 train_acc=0.9626 val_loss=0.1485 val_acc=0.9608
Epoch 004 | train_loss=0.1373 train_acc=0.9696 val_loss=0.1213 val_acc=0.9642
Epoch 004 | train_loss=0.1373 train_acc=0.9696 val_loss=0.1213 val_acc=0.9642
Epoch 005 | train_loss=0.1153 train_acc=0.9742 val_loss=0.1061 val_acc=0.9672
Epoch 005 | train_loss=0.1153 train_acc=0.9742 val_loss=0.1061 val_acc=0.9672
Epoch 006 | train_loss=0.1018 train_acc=0.9775 val_loss=0.0966 val_acc=0.9688
Epoch 006 | train_loss=0.1018 train_acc=0.9775 val_loss=0.0966 val_acc=0.9688
Epoch 007 | train_loss=0.0924 train_acc=0.9798 val_loss=0.0901 val_acc=0.9695
Epoch 007 | train_loss=0.0924 train_acc=0.9798 val_loss=0.0901 v

As seen from the report, the char n-gram neural network model outperforms the word n-gram neural in the test set evaluations. This can be attributed to the fact that character n-grams are more granular and are able to considerably represent the style of the text [8]. This is a surprising result considering that there is more similarity on character n-grams across the different authors as seen in the corpus analysis. It is also interesting to see that shallow (1 layer) neural network was considered to be the best configuration. This might be due to having a relatively small training dataset, as deep neural networks does well when given large training datasets [9]

[1]
Modupe, A., Celik, T., Marivate, V., & Olugbara, O. O. (2022). Post-authorship attribution using regularized deep neural network. Applied Sciences, 12(15), 7518.

[2]
Saha, N., Das, P., & Saha, H. N. (2018). Authorship attribution of short texts using multi-layer perceptron. International Journal of Applied Pattern Recognition, 5(3), 251-259.

[3]
Sari, Y., Vlachos, A., & Stevenson, M. (2017, April). Continuous n-gram representations for authorship attribution. In Proceedings of the 15th conference of the European chapter of the association for computational linguistics: Volume 2, short papers (pp. 267-273).

[4]
GeeksforGeeks. (2025, March 11). ReLU activation function in deep learning. https://www.geeksforgeeks.org/deep-learning/relu-activation-function-in-deep-learning/

[5]
Kingma, D. P., & Ba, J. (2014). Adam: A method for stochastic optimization. arXiv preprint arXiv:1412.6980.

[6]
GeeksforGeeks. (2025, August 28). Regularization by early stopping. https://www.geeksforgeeks.org/machine-learning/regularization-by-early-stopping/

[7]
GeeksforGeeks. (2025, March 6). Classification using scikit‑learn multi‑layer perceptron. https://www.geeksforgeeks.org/machine-learning/classification-using-sklearn-multi-layer-perceptron/

[8]
Houvardas, J., & Stamatatos, E. (2006, September). N-gram feature selection for authorship identification. In International conference on artificial intelligence: Methodology, systems, and applications (pp. 77-86). Berlin, Heidelberg: Springer Berlin Heidelberg.

[9]
GeeksforGeeks. (2025, April 17). Shallow neural networks. https://www.geeksforgeeks.org/deep-learning/shallow-neural-networks/